In [1]:
import numpy as np
import random

In [2]:
class Value:
    def __init__(self, value, children = (), op = '', label = '', grad = 0):
        
        self.value = value
        self._prev = children
        self._backward = lambda : None
        self.grad = grad
        self._op = op
        self.label = label

    def __repr__(self):
        
        return f'{self.value}'

    def __add__(self, other):

        if isinstance(other, Value):
            new_val = other.value

            out = Value(new_val + self.value, (self, other), '+')
            
            def _backward():
                self.grad += 1.0 * out.grad
                other.grad += 1.0 * out.grad 
            out._backward = _backward
            
        else:
            return NotImplemented
        
        return out

    def __mul__(self, other):
        
        if isinstance(other, Value):
            new_val = other.value

            out = Value(new_val * self.value, (self, other), '*')
            
            def _backward():
                self.grad += out.grad * other.value
                other.grad += out.grad * self.value
            out._backward = _backward
            
        else:
            return NotImplemented
        
        return out

    def tanh(self):
        x = self.value
        t = (np.exp(2*x) - 1) / (np.exp(2*x) + 1)
        out = Value(t, (self, ), 'tanh')

        def _backward():
            self.grad += out.grad * (1.0 - t**2)
        out._backward = _backward
        
        return out

    def backward(self):
        
        topo = []
        visited = set()
        
        def dfs(node):
    
            if node in visited:
                return
    
            visited.add(node)
    
            for child in node._prev:
                dfs(child)
    
            topo.append(node)
    
        dfs(self)
    
        self.grad = 1

        for node in reversed(topo):
            node._backward()

In [3]:
class Neuron:

    def __init__(self, n):
        self.w = [Value(random.uniform(-1,1)) for _ in range(n)]
        self.b = Value(random.uniform(-1,1))

    def __call__(self, x):
        aggr = sum((xi*wi for xi, wi in zip(x, self.w)), self.b)
        activated = aggr.tanh()
        return aggr

    def __repr__(self):
        return f'Weights: {[w for w in self.w]}, Bias: {self.b}'

In [4]:
class Layer:

    def __init__(self, inp, out):
        self.inp = inp
        self.out = out
        self.neurons = [Neuron(self.inp) for _ in range(self.out)]

    def __call__(self, x):
        out = [neuron(x) for neuron in self.neurons]
        return out

    def __repr__(self):
        out_string = [f"Neuron : {neuron}" for neuron in self.neurons]
        return ',\n'.join(out_string)

In [5]:
class MLP:

    def __init__(self, ins, outs):
        sizes = [ins] + outs
        self.layers = [Layer(sizes[i-1], sizes[i]) for i in range(1, len(sizes))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)

        return x

    def __repr__(self):
        out_string = [f'Layer{i}:\n{layer}' for i, layer in enumerate(self.layers, start = 1)]
        return ',\n\n'.join(out_string)

In [6]:
x = [Value(i) for i in range(1,4)]
x

[1, 2, 3]

In [7]:
ins = 3
outs = [4,3,1]

In [8]:
model = MLP(ins, outs)
model

Layer1:
Neuron : Weights: [-0.9017511101315805, 0.5158997900619842, 0.7514707588769169], Bias: 0.5565041844753642,
Neuron : Weights: [0.8137315745882234, -0.12370623214004994, 0.39444379942374086], Bias: -0.021237067398895526,
Neuron : Weights: [0.22544009791582198, 0.20855477092619767, 0.11933179904132674], Bias: 0.9937688393150219,
Neuron : Weights: [0.37959830574873576, -0.10045202974035083, -0.050134651284296616], Bias: 0.9105073757847653,

Layer2:
Neuron : Weights: [-0.28383494002825094, 0.8412270565678952, 0.007539585444455099, 0.5068405555278133], Bias: 0.10851612112741549,
Neuron : Weights: [-0.5738455942376, -0.9475840494912904, -0.8330632031860539, -0.6238672580969544], Bias: 0.5227701709942403,
Neuron : Weights: [-0.2111345603834034, 0.6297351420792225, -0.37874320989309096, 0.8587303629572878], Bias: 0.27463982047281177,

Layer3:
Neuron : Weights: [0.04886162216459655, 0.17429843720615712, -0.8585353086285887], Bias: 0.10252670322419477

In [9]:
pred = model(x)
pred

[-1.3989039795237503]